# Tutorial: Linear Probing

## Setup
**Model:** `distilbert-base-uncased`

**Task:** IMDb Sentiment Classification

## Learning Objectives

By the end of this notebook you will be able to:

1. **Explain what a probe is** and what probing accuracy tells us about pretrained representations.
2. **Implement linear probing**: freeze an entire transformer backbone and train only a small classification head on top.


This notebook is designed to run on **Google Colab** with a free T4 GPU.  
Go to `Runtime → Change runtime type → T4 GPU` before running.

## 1. Setup

In [ ]:
# Install required libraries
# (peft is intentionally omitted — probing needs no extra libraries)
!pip install -q transformers datasets torch accelerate

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
import numpy as np
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cpu":
    print("WARNING: No GPU detected. Training will be slow.")
    print("Go to Runtime > Change runtime type > GPU to enable it.")

## 2. What is a Probe?

A model like `distilbert-base-uncased` was pretrained on **masked language modeling (MLM)** — it learned to predict missing words in sentences. It was never shown a sentiment label during pretraining. And yet, it learned rich representations of language.

The procedure is simple:
1. **Freeze** all backbone weights (no gradient flows into DistilBERT).
2. **Extract** fixed representations from the backbone.
3. **Train** a small classifier (often a single linear layer) on those representations.
4. **Measure accuracy** of that classifier.

**Interpretation:**
- **High probing accuracy** → the property (e.g., sentiment) is **linearly encoded** in the frozen representations. The model "knows" about it even though it wasn't trained for it.
- **Low probing accuracy** → the property is not easily readable from these representations (either not encoded, or encoded non-linearly).

Probing is, in a sense, a partial fine-tuning.

| | Probing | Full Fine-tuning |
|---|---|---|
| Backbone weights | **Frozen** | Updated |
| Trainable params | ~1,500 | ~66 million |

## 3. Load and Subsample the IMDb Dataset

In [ ]:
# Load IMDb and subsample for speed
# Full dataset: 25,000 train + 25,000 test — we use a small subset
raw = load_dataset("imdb")
train_data = raw["train"].shuffle(seed=42).select(range(2000))
test_data  = raw["test"].shuffle(seed=42).select(range(500))

print(f"Train size: {len(train_data)}, Test size: {len(test_data)}")
print("\nSample review (truncated):")
print(train_data[0]["text"][:300], "...")
print("Label:", "POSITIVE" if train_data[0]["label"] == 1 else "NEGATIVE")

In [ ]:
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=256  # 256 instead of 512 to save memory and time
    )

train_tok = train_data.map(tokenize, batched=True, remove_columns=["text"])
test_tok  = test_data.map(tokenize,  batched=True, remove_columns=["text"])

train_tok.set_format("torch")
test_tok.set_format("torch")

train_loader = DataLoader(train_tok, batch_size=32, shuffle=True)
test_loader  = DataLoader(test_tok,  batch_size=64)

print(f"Train batches: {len(train_loader)}, Test batches: {len(test_loader)}")

## 4. The `[CLS]` Token as a Sentence Representation

DistilBERT (like BERT) always prepends a special `[CLS]` token to every input sequence. After passing through all transformer layers, the hidden state at **position 0** (the `[CLS]` position) serves as a summary representation of the entire sequence.

```
Input:   [CLS]  "This"  "movie"  "was"  "great"  [SEP]
Index:    0       1       2        3       4        5
```

We extract it with: `output.last_hidden_state[:, 0, :]`

- `last_hidden_state` has shape `(batch_size, sequence_length, 768)`
- `[:, 0, :]` selects position 0 (the `[CLS]` token) for every example in the batch
- Result shape: `(batch_size, 768)` — a 768-dimensional vector per example

> **Why 768?** DistilBERT's hidden size is 768. This is the dimensionality of all hidden states throughout the network.

## 5. Build the Linear Probe Model

In [ ]:
# Load the backbone (encoder only, no classification head)
backbone = AutoModel.from_pretrained(MODEL_NAME).to(device)

# ── FREEZE ALL backbone parameters ──────────────────────────────────────────
# This is the defining step of probing: no gradient will ever flow into
# DistilBERT. We are only reading out its representations.
for param in backbone.parameters():
    param.requires_grad = False

# Put backbone in eval mode to disable dropout (deterministic representations)
# If we left it in train mode, representations would randomly vary each pass,
# making the probe harder to train and results harder to interpret.
backbone.eval()

total_params = sum(p.numel() for p in backbone.parameters())
print(f"Backbone total params (all frozen): {total_params:,}")

In [ ]:
class LinearProbe(nn.Module):
    """A single linear layer that classifies from [CLS] embeddings."""

    def __init__(self, hidden_size=768, num_classes=2):
        super().__init__()
        self.classifier = nn.Linear(hidden_size, num_classes)

    def forward(self, backbone_output):
        # Extract the [CLS] token hidden state: shape (batch, 768)
        cls_embedding = backbone_output.last_hidden_state[:, 0, :]
        return self.classifier(cls_embedding)


probe = LinearProbe().to(device)

trainable_params = sum(p.numel() for p in probe.parameters() if p.requires_grad)
print(f"Trainable probe params: {trainable_params:,}")
# Expected: 768 * 2 (weights) + 2 (bias) = 1,538
print("\nBreakdown:")
for name, p in probe.named_parameters():
    print(f"  {name}: shape {list(p.shape)} → {p.numel()} params")

## 6. Training the Probe

Two important implementation details:

1. **`torch.no_grad()` around the backbone call** — The backbone is frozen, so we don't need to compute or store gradients for its operations. Without this, PyTorch would still build a computation graph for the backbone, wasting ~2× memory and time on Colab.

2. **Optimizer only receives `probe.parameters()`** — Even if we forgot to freeze the backbone, the optimizer would not update it since it only sees the probe's parameters.

In [ ]:
optimizer = torch.optim.AdamW(probe.parameters(), lr=1e-3)
criterion = nn.CrossEntropyLoss()
EPOCHS = 3

for epoch in range(EPOCHS):
    probe.train()
    # backbone stays in eval() — already set above

    total_loss, correct, total = 0.0, 0, 0

    for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        # Frozen backbone: use no_grad to skip gradient computation entirely
        with torch.no_grad():
            backbone_output = backbone(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

        # Only the probe's forward pass is in the computation graph
        logits = probe(backbone_output)
        loss   = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()   # gradients only flow into probe.classifier weights
        optimizer.step()

        total_loss += loss.item()
        correct    += (logits.argmax(dim=-1) == labels).sum().item()
        total      += labels.size(0)

    avg_loss = total_loss / len(train_loader)
    acc      = correct / total
    print(f"  → Loss: {avg_loss:.4f} | Train Acc: {acc:.4f}")

## 7. Evaluate the Linear Probe

In [ ]:
probe.eval()
correct, total = 0, 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="Evaluating"):
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["label"].to(device)

        backbone_output = backbone(input_ids=input_ids, attention_mask=attention_mask)
        logits          = probe(backbone_output)

        correct += (logits.argmax(dim=-1) == labels).sum().item()
        total   += labels.size(0)

final_acc = correct / total
print(f"\nLinear Probe Test Accuracy: {final_acc:.4f} ({final_acc*100:.1f}%)")

## 8. Interpreting the Results

A linear probe with **~80% accuracy** was trained with only 1,538 parameters — a single linear layer — on top of a completely frozen DistilBERT backbone.

**What this tells us:**

The `[CLS]` representations produced by DistilBERT are **linearly separable** for sentiment. This is remarkable because:
- DistilBERT was trained to predict masked words, not to classify sentiment.
- The probe accuracy is an emergent consequence of language modeling: to predict that "*The movie was \[MASK\]*" is likely completed by "great" rather than "terrible", the model must build representations that distinguish these different contexts — which implicitly encodes sentiment.

**Ceiling of probing:**
- The probe accuracy will always be ≤ (full) fine-tuning accuracy, because fine-tuning is strictly more expressive — it can reshape the representations, not just read them.
- A large gap between probing and fine-tuning suggests the model *has* the relevant information but it is not linearly organized.